In [1]:
# -------------------------------------------------------
# CELL 1 - Imports and config
# -------------------------------------------------------
from pyspark.sql.functions import (
    col, trim, lower, upper, when, lit, 
    to_timestamp, current_timestamp,
    regexp_replace, explode, from_json
)
from pyspark.sql.types import *
from datetime import datetime, timezone
import uuid

PIPELINE_RUN_ID = str(uuid.uuid4())
LAYER = "silver"

print(f"Silver transformation run: {PIPELINE_RUN_ID}")
print(f"Timestamp: {datetime.now(timezone.utc)}")

StatementMeta(, bf7a20a1-cff6-492f-b375-e5a82f9b718c, 3, Finished, Available, Finished, False)

Silver transformation run: 39b0cf2c-4e3c-4a68-b175-2256627d8e76
Timestamp: 2026-05-10 13:08:18.872158+00:00


In [2]:
# -------------------------------------------------------
# CELL 2 - bronze_otx_pulses → silver_otx_pulses
# -------------------------------------------------------
df_bronze_pulses = spark.sql("SELECT * FROM bronze_otx_pulses")

df_silver_pulses = (df_bronze_pulses
    # clean strings
    .withColumn("pulse_name", trim(col("pulse_name")))
    .withColumn("author_name", trim(col("author_name")))
    .withColumn("description", trim(col("description")))
    
    # cast timestamps
    .withColumn("created", to_timestamp(col("created")))
    .withColumn("modified", to_timestamp(col("modified")))
    .withColumn("ingested_at", to_timestamp(col("ingested_at")))
    
    # standardise TLP to uppercase
    .withColumn("tlp", upper(trim(col("tlp"))))
    
    # standardise nulls - replace empty strings with None
    .withColumn("description", 
        when(col("description") == "", None)
        .otherwise(col("description")))
    
    # add silver metadata
    .withColumn("silver_processed_at", current_timestamp())
    .withColumn("pipeline_run_id", lit(PIPELINE_RUN_ID))
    
    # drop rows with no pulse_id - critical field
    .filter(col("pulse_id").isNotNull() & (col("pulse_id") != ""))
)

print(f"Bronze pulses: {df_bronze_pulses.count()} rows")
print(f"Silver pulses: {df_silver_pulses.count()} rows")
print(f"Dropped: {df_bronze_pulses.count() - df_silver_pulses.count()} rows")
df_silver_pulses.printSchema()

StatementMeta(, bf7a20a1-cff6-492f-b375-e5a82f9b718c, 4, Finished, Available, Finished, False)

Bronze pulses: 250 rows
Silver pulses: 250 rows
Dropped: 0 rows
root
 |-- author_name: string (nullable = true)
 |-- created: timestamp (nullable = true)
 |-- description: string (nullable = true)
 |-- indicator_count: long (nullable = true)
 |-- ingested_at: timestamp (nullable = true)
 |-- malware_families: string (nullable = true)
 |-- modified: timestamp (nullable = true)
 |-- pipeline_run_id: string (nullable = false)
 |-- pulse_id: string (nullable = true)
 |-- pulse_name: string (nullable = true)
 |-- source_system: string (nullable = true)
 |-- tags: string (nullable = true)
 |-- targeted_countries: string (nullable = true)
 |-- tlp: string (nullable = true)
 |-- silver_processed_at: timestamp (nullable = false)



In [3]:
# -------------------------------------------------------
# CELL 3 - Write silver_otx_pulses
# -------------------------------------------------------
(df_silver_pulses.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("silver_otx_pulses")
)

count = spark.sql("SELECT COUNT(*) as total FROM silver_otx_pulses").collect()[0]["total"]
print(f"silver_otx_pulses: {count} rows written successfully")

StatementMeta(, bf7a20a1-cff6-492f-b375-e5a82f9b718c, 5, Finished, Available, Finished, False)

silver_otx_pulses: 250 rows written successfully


In [4]:
# -------------------------------------------------------
# CELL 4 - bronze_otx_indicators → silver_otx_indicators
# -------------------------------------------------------
df_bronze_indicators = spark.sql("SELECT * FROM bronze_otx_indicators")

df_silver_indicators = (df_bronze_indicators
    # clean strings
    .withColumn("indicator", trim(lower(col("indicator"))))
    .withColumn("type", trim(col("type")))
    .withColumn("title", trim(col("title")))
    .withColumn("description",
        when(col("description") == "", None)
        .otherwise(trim(col("description"))))

    # cast timestamps
    .withColumn("created", to_timestamp(col("created")))
    .withColumn("ingested_at", to_timestamp(col("ingested_at")))
    .withColumn("expiration",
        when(col("expiration") == "None", None)
        .otherwise(to_timestamp(col("expiration"))))

    # flag correlation-priority indicator types
    # these are the types we match against internal firewall logs
    .withColumn("is_correlation_target",
        when(col("type").isin(["domain", "hostname", "URL", "IPv4"]), True)
        .otherwise(False))

    # cast is_active to boolean
    .withColumn("is_active", col("is_active").cast("boolean"))

    # add silver metadata
    .withColumn("silver_processed_at", current_timestamp())
    .withColumn("pipeline_run_id", lit(PIPELINE_RUN_ID))

    # drop rows with no indicator value
    .filter(col("indicator").isNotNull() & (col("indicator") != ""))
)

print(f"Bronze indicators: {df_bronze_indicators.count()} rows")
print(f"Silver indicators: {df_silver_indicators.count()} rows")
print(f"Dropped: {df_bronze_indicators.count() - df_silver_indicators.count()} rows")
print(f"\nCorrelation targets: {df_silver_indicators.filter(col('is_correlation_target') == True).count()} rows")
df_silver_indicators.printSchema()

StatementMeta(, bf7a20a1-cff6-492f-b375-e5a82f9b718c, 6, Finished, Available, Finished, False)

Bronze indicators: 6881 rows
Silver indicators: 6881 rows
Dropped: 0 rows

Correlation targets: 2535 rows
root
 |-- created: timestamp (nullable = true)
 |-- description: string (nullable = true)
 |-- expiration: timestamp (nullable = true)
 |-- indicator: string (nullable = true)
 |-- indicator_id: string (nullable = true)
 |-- ingested_at: timestamp (nullable = true)
 |-- is_active: boolean (nullable = true)
 |-- pipeline_run_id: string (nullable = false)
 |-- pulse_id: string (nullable = true)
 |-- source_system: string (nullable = true)
 |-- title: string (nullable = true)
 |-- type: string (nullable = true)
 |-- is_correlation_target: boolean (nullable = false)
 |-- silver_processed_at: timestamp (nullable = false)



In [5]:
# -------------------------------------------------------
# CELL 5 - Write silver_otx_indicators
# -------------------------------------------------------
(df_silver_indicators.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("silver_otx_indicators")
)

count = spark.sql("SELECT COUNT(*) as total FROM silver_otx_indicators").collect()[0]["total"]
print(f"silver_otx_indicators: {count} rows written successfully")

StatementMeta(, bf7a20a1-cff6-492f-b375-e5a82f9b718c, 7, Finished, Available, Finished, False)

silver_otx_indicators: 6881 rows written successfully


In [6]:
# -------------------------------------------------------
# CELL 6 - bronze_firewall_logs → silver_firewall_logs
# -------------------------------------------------------
df_bronze_firewall = spark.sql("SELECT * FROM bronze_firewall_logs")

df_silver_firewall = (df_bronze_firewall
    # clean strings
    .withColumn("internal_ip", trim(col("internal_ip")))
    .withColumn("destination", trim(lower(col("destination"))))
    .withColumn("destination_type", trim(col("destination_type")))
    .withColumn("department", trim(col("department")))

    # cast timestamp
    .withColumn("timestamp", to_timestamp(col("timestamp")))
    .withColumn("ingested_at", to_timestamp(col("ingested_at")))

    # standardise action to lowercase
    .withColumn("action", lower(trim(col("action"))))

    # standardise severity to lowercase
    .withColumn("severity", lower(trim(col("severity"))))

    # add numeric severity score for ranking
    # low=1, medium=2, high=3
    .withColumn("severity_score",
        when(col("severity") == "high", 3)
        .when(col("severity") == "medium", 2)
        .when(col("severity") == "low", 1)
        .otherwise(0))

    # flag rows where action was blocked
    .withColumn("is_blocked",
        when(col("action") == "blocked", True)
        .otherwise(False))

    # add silver metadata
    .withColumn("silver_processed_at", current_timestamp())
    .withColumn("pipeline_run_id", lit(PIPELINE_RUN_ID))

    # drop rows with no log_id
    .filter(col("log_id").isNotNull() & (col("log_id") != ""))
)

print(f"Bronze firewall: {df_bronze_firewall.count()} rows")
print(f"Silver firewall: {df_silver_firewall.count()} rows")
print(f"Dropped: {df_bronze_firewall.count() - df_silver_firewall.count()} rows")
print(f"\nBlocked events: {df_silver_firewall.filter(col('is_blocked') == True).count()}")
print(f"High severity: {df_silver_firewall.filter(col('severity') == 'high').count()}")
df_silver_firewall.printSchema()

StatementMeta(, bf7a20a1-cff6-492f-b375-e5a82f9b718c, 8, Finished, Available, Finished, False)

Bronze firewall: 1000 rows
Silver firewall: 1000 rows
Dropped: 0 rows

Blocked events: 262
High severity: 63
root
 |-- log_id: string (nullable = true)
 |-- timestamp: timestamp (nullable = true)
 |-- internal_ip: string (nullable = true)
 |-- destination: string (nullable = true)
 |-- destination_type: string (nullable = true)
 |-- action: string (nullable = true)
 |-- bytes_transferred: long (nullable = true)
 |-- department: string (nullable = true)
 |-- severity: string (nullable = true)
 |-- ingested_at: timestamp (nullable = true)
 |-- source_system: string (nullable = true)
 |-- pipeline_run_id: string (nullable = false)
 |-- severity_score: integer (nullable = false)
 |-- is_blocked: boolean (nullable = false)
 |-- silver_processed_at: timestamp (nullable = false)



In [7]:
# -------------------------------------------------------
# CELL 7 - Write silver_firewall_logs
# -------------------------------------------------------
(df_silver_firewall.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("silver_firewall_logs")
)

count = spark.sql("SELECT COUNT(*) as total FROM silver_firewall_logs").collect()[0]["total"]
print(f"silver_firewall_logs: {count} rows written successfully")

StatementMeta(, bf7a20a1-cff6-492f-b375-e5a82f9b718c, 9, Finished, Available, Finished, False)

silver_firewall_logs: 1000 rows written successfully


In [8]:
# -------------------------------------------------------
# CELL 8 - Silver layer verification
# -------------------------------------------------------
tables = ["silver_otx_pulses", "silver_otx_indicators", "silver_firewall_logs"]

print("=== Silver Layer Verification ===\n")
for table in tables:
    count = spark.sql(f"SELECT COUNT(*) as total FROM {table}").collect()[0]["total"]
    print(f"✓ {table}: {count} rows")

print("\n=== Data Quality Check: Nulls in critical fields ===")
spark.sql("""
    SELECT 
        SUM(CASE WHEN pulse_id IS NULL THEN 1 ELSE 0 END) as null_pulse_ids,
        SUM(CASE WHEN pulse_name IS NULL THEN 1 ELSE 0 END) as null_pulse_names,
        SUM(CASE WHEN created IS NULL THEN 1 ELSE 0 END) as null_created
    FROM silver_otx_pulses
""").show()

spark.sql("""
    SELECT 
        SUM(CASE WHEN indicator IS NULL THEN 1 ELSE 0 END) as null_indicators,
        SUM(CASE WHEN pulse_id IS NULL THEN 1 ELSE 0 END) as null_pulse_ids,
        SUM(CASE WHEN type IS NULL THEN 1 ELSE 0 END) as null_types
    FROM silver_otx_indicators
""").show()

spark.sql("""
    SELECT 
        SUM(CASE WHEN log_id IS NULL THEN 1 ELSE 0 END) as null_log_ids,
        SUM(CASE WHEN destination IS NULL THEN 1 ELSE 0 END) as null_destinations,
        SUM(CASE WHEN timestamp IS NULL THEN 1 ELSE 0 END) as null_timestamps
    FROM silver_firewall_logs
""").show()

StatementMeta(, bf7a20a1-cff6-492f-b375-e5a82f9b718c, 10, Finished, Available, Finished, False)

=== Silver Layer Verification ===

✓ silver_otx_pulses: 250 rows
✓ silver_otx_indicators: 6881 rows
✓ silver_firewall_logs: 1000 rows

=== Data Quality Check: Nulls in critical fields ===
+--------------+----------------+------------+
|null_pulse_ids|null_pulse_names|null_created|
+--------------+----------------+------------+
|             0|               0|           0|
+--------------+----------------+------------+

+---------------+--------------+----------+
|null_indicators|null_pulse_ids|null_types|
+---------------+--------------+----------+
|              0|             0|         0|
+---------------+--------------+----------+

+------------+-----------------+---------------+
|null_log_ids|null_destinations|null_timestamps|
+------------+-----------------+---------------+
|           0|                0|              0|
+------------+-----------------+---------------+

